# Trabajo 01 — Parte 1: Funciones de Prueba y Descenso por Gradiente

**Curso:** Optimización | **Profesor:** Juan David Ospina Arango  
**Entrega:** 24 de marzo de 2026

En este notebook se estudian dos funciones clásicas de prueba para optimización y se aplica el método de **descenso por gradiente** con búsqueda de línea por retroceso (*backtracking line search*) en 2D y 3D.

| # | Sección | Contenido |
|---|---|---|
| 1 | Funciones de prueba | Definición matemática, gradientes analíticos, verificación |
| 2 | Visualización | Contour plots 2D y superficies 3D |
| 3 | Descenso por gradiente | Implementación con backtracking, resultados 2D y 3D |
| 4 | Animaciones | GIFs del proceso de convergencia |
| 5 | Conclusiones | Hallazgos clave |

In [ ]:
# Descomentar si se ejecuta en Google Colab
# !pip install numpy matplotlib pillow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from pathlib import Path
from IPython.display import Image, display

# --- Constantes globales ---
SEED        = 42
BOUNDS_2D   = (-3.0, 3.0)   # dominio de visualización
BOUNDS_OPT  = (-5.0, 5.0)   # dominio de optimización
GRID_N      = 300            # resolución del grid de contour
A_RASTRIGIN = 10             # parámetro A de la función de Rastrigin
GIF_FPS     = 20
GIF_FRAMES  = 150

# Parámetros del descenso por gradiente
GD_ALPHA0   = 1.0
GD_RHO      = 0.5
GD_C_ARMIJO = 1e-4
GD_TOL      = 1e-6
GD_MAX_ITER = 10_000

np.random.seed(SEED)
Path('outputs').mkdir(exist_ok=True)
print('Entorno listo.')

## 1. Funciones de prueba

Se seleccionaron dos funciones ampliamente usadas como *benchmarks* en optimización global:

### 1.1 Función de Rosenbrock

$$f(\mathbf{x}) = \sum_{i=1}^{n-1} \left[ 100\,(x_{i+1} - x_i^2)^2 + (1 - x_i)^2 \right]$$

- **Mínimo global:** $f(1, 1, \ldots, 1) = 0$
- **Dificultad:** el mínimo yace en el fondo de un valle estrecho y curvado; el gradiente apunta casi perpendicular a la dirección de menor valor.

### 1.2 Función de Rastrigin

$$f(\mathbf{x}) = An + \sum_{i=1}^{n} \left[ x_i^2 - A\cos(2\pi x_i) \right], \quad A = 10$$

- **Mínimo global:** $f(0, 0, \ldots, 0) = 0$
- **Dificultad:** altamente multimodal — contiene $O(10^n)$ mínimos locales en cuadrícula regular, lo que atrapa al gradiente con facilidad.

In [ ]:
def rosenbrock(x):
    """Rosenbrock: mínimo global en x=[1,...,1], f=0."""
    x = np.asarray(x, dtype=float)
    return float(np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1.0 - x[:-1])**2))

def grad_rosenbrock(x):
    """Gradiente analítico de Rosenbrock."""
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    g[:-1] += -400.0 * x[:-1] * (x[1:] - x[:-1]**2) - 2.0 * (1.0 - x[:-1])
    g[1:]  +=  200.0 * (x[1:] - x[:-1]**2)
    return g

def rastrigin(x):
    """Rastrigin: mínimo global en x=[0,...,0], f=0."""
    x = np.asarray(x, dtype=float)
    return float(A_RASTRIGIN * len(x) + np.sum(x**2 - A_RASTRIGIN * np.cos(2*np.pi*x)))

def grad_rastrigin(x):
    """Gradiente analítico de Rastrigin."""
    x = np.asarray(x, dtype=float)
    return 2.0 * x + 2.0 * np.pi * A_RASTRIGIN * np.sin(2*np.pi*x)

# Verificación en el mínimo conocido
assert abs(rosenbrock([1.0, 1.0])) < 1e-10
assert abs(rosenbrock([1.0, 1.0, 1.0])) < 1e-10
assert abs(rastrigin([0.0, 0.0])) < 1e-10
assert abs(rastrigin([0.0, 0.0, 0.0])) < 1e-10
print('Verificación OK — ambas funciones evaluadas correctamente en sus mínimos.')

### 1.2 Funciones adicionales

La tarea requiere seis funciones en total. Se agregan las cuatro restantes:

| Funcion | Dominio | Minimo global | Dimensiones |
|---|---|---|---|
| **Schwefel** | $[-500, 500]^n$ | $f(420.97,\ldots) = 0$ | 2D y 3D |
| **Griewank** | $[-600, 600]^n$ | $f(0,\ldots,0) = 0$ | 2D y 3D |
| **Goldstein-Price** | $[-2, 2]^2$ | $f(0,-1) = 3$ | Solo 2D |
| **Camel 6-hump** | $x_1\in[-3,3],\, x_2\in[-2,2]$ | $f(\pm0.0898, \mp0.7126)\approx -1.0316$ | Solo 2D |

> Goldstein-Price y Camel son funciones **exclusivamente bidimensionales**.
> Schwefel y Griewank se extienden a 3D de forma natural.

In [ ]:
# ---------------------------------------------------------------------------
# Funciones adicionales requeridas por la tarea
# Cada funcion incluye su formula, dominio y minimo global conocido.
# ---------------------------------------------------------------------------

def schwefel(x):
    """
    Funcion de Schwefel.
    Dominio: x_i en [-500, 500].
    Minimo global: f(420.9687, ...) = 0.
    Nota: el minimo esta lejos del centro del dominio, lo que engana facilmente al GD.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    return float(418.9829 * n - np.sum(x * np.sin(np.sqrt(np.abs(x)))))


def grad_schwefel(x):
    """Gradiente de Schwefel por diferencias finitas centradas."""
    return gradiente_numerico(schwefel, x)


def griewank(x):
    """
    Funcion de Griewank.
    Dominio: x_i en [-600, 600].
    Minimo global: f(0, ..., 0) = 0.
    Nota: tiene muchos minimos locales uniformemente distribuidos.
    """
    x = np.asarray(x, dtype=float)
    suma = np.sum(x**2) / 4000
    prod = np.prod(np.cos(x / np.sqrt(np.arange(1, len(x) + 1))))
    return float(1 + suma - prod)


def grad_griewank(x):
    """Gradiente de Griewank por diferencias finitas centradas."""
    return gradiente_numerico(griewank, x)


def goldstein_price(x):
    """
    Funcion de Goldstein-Price (solo 2D).
    Dominio: x_i en [-2, 2].
    Minimo global: f(0, -1) = 3.
    Nota: paisaje muy irregular, multiple minimos locales.
    """
    x = np.asarray(x, dtype=float)
    x1, x2 = float(x[0]), float(x[1])
    a = (1 + (x1 + x2 + 1)**2
         * (19 - 14*x1 + 3*x1**2 - 14*x2 + 6*x1*x2 + 3*x2**2))
    b = (30 + (2*x1 - 3*x2)**2
         * (18 - 32*x1 + 12*x1**2 + 48*x2 - 36*x1*x2 + 27*x2**2))
    return float(a * b)


def grad_goldstein_price(x):
    """Gradiente de Goldstein-Price por diferencias finitas centradas."""
    return gradiente_numerico(goldstein_price, x)


def camel_6hump(x):
    """
    Funcion de las seis jorobas de camello (Six-Hump Camel), solo 2D.
    Dominio: x1 en [-3, 3], x2 en [-2, 2].
    Minimos globales: f(0.0898, -0.7126) = f(-0.0898, 0.7126) ~ -1.0316.
    Nota: dos minimos globales simetricos, seis jorobas locales.
    """
    x = np.asarray(x, dtype=float)
    x1, x2 = float(x[0]), float(x[1])
    return float(
        (4 - 2.1*x1**2 + x1**4 / 3) * x1**2
        + x1 * x2
        + (-4 + 4*x2**2) * x2**2
    )


def grad_camel_6hump(x):
    """Gradiente de Camel 6-hump por diferencias finitas centradas."""
    return gradiente_numerico(camel_6hump, x)


# ---------------------------------------------------------------------------
# Verificacion en los minimos conocidos
# ---------------------------------------------------------------------------
print("Verificacion de nuevas funciones en sus minimos globales:")
print(f"  Schwefel       [420.97, 420.97] -> f = {schwefel([420.9687, 420.9687]):.4f}  (esperado ~ 0)")
print(f"  Griewank       [0, 0]           -> f = {griewank([0.0, 0.0]):.4f}             (esperado = 0)")
print(f"  Goldstein      [0, -1]          -> f = {goldstein_price([0.0, -1.0]):.4f}     (esperado = 3)")
print(f"  Camel 6-hump   [0.0898,-0.7126] -> f = {camel_6hump([0.0898, -0.7126]):.4f}  (esperado ~ -1.0316)")

### 1.3 Verificación de gradientes por diferencias finitas

Se compara el gradiente analítico contra la aproximación numérica centrada:

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x + \varepsilon e_i) - f(x - \varepsilon e_i)}{2\varepsilon}, \quad \varepsilon = 10^{-5}$$

In [ ]:
def gradiente_numerico(f, x, eps=1e-5):
    """Gradiente por diferencias finitas centradas."""
    g = np.zeros_like(x)
    for i in range(len(x)):
        xp, xm = x.copy(), x.copy()
        xp[i] += eps
        xm[i] -= eps
        g[i] = (f(xp) - f(xm)) / (2 * eps)
    return g

x_test = np.random.uniform(-2, 2, 3)
for nombre, f, gf in [('Rosenbrock', rosenbrock, grad_rosenbrock),
                       ('Rastrigin',  rastrigin,  grad_rastrigin)]:
    error = np.max(np.abs(gf(x_test) - gradiente_numerico(f, x_test)))
    print(f'{nombre}: error máx. gradiente = {error:.2e}  {"OK" if error < 1e-4 else "FALLA"}')

## 2. Visualización 2D y 3D

Se generan contour plots (vista cenital) y superficies 3D para entender la geometría antes de optimizar.
Se aplica $\log(1 + f)$ al contour para mejorar el contraste en regiones con valores muy distintos.

Los grids se calculan **una sola vez** y se reutilizan en todas las visualizaciones y animaciones.

In [ ]:
# ---------------------------------------------------------------------------
# Lista canonica de funciones usada en todo el notebook.
# Se distingue entre funciones validas en nD y las que son solo 2D.
# ---------------------------------------------------------------------------

# Funciones validas en 2D y 3D
CONFIGS_ND = [
    ('Rosenbrock', rosenbrock,       grad_rosenbrock),
    ('Rastrigin',  rastrigin,        grad_rastrigin),
    ('Schwefel',   schwefel,         grad_schwefel),
    ('Griewank',   griewank,         grad_griewank),
]

# Funciones exclusivamente 2D por definicion matematica
CONFIGS_2D_ONLY = [
    ('Goldstein-Price', goldstein_price,  grad_goldstein_price),
    ('Camel 6-hump',    camel_6hump,      grad_camel_6hump),
]

# CONFIGS: todas las funciones (para visualizacion y GD en 2D)
CONFIGS = CONFIGS_ND + CONFIGS_2D_ONLY


def make_grid(f, bounds=BOUNDS_2D, n=GRID_N):
    """Evalua f en un grid 2D para visualizacion."""
    x  = np.linspace(*bounds, n)
    X, Y = np.meshgrid(x, x)
    XY = np.stack([X, Y], axis=-1)
    Z  = np.apply_along_axis(f, 2, XY)
    return X, Y, Z


# Pre-calcular grids una sola vez (todos son 2D)
grids = {nombre: make_grid(f) for nombre, f, _ in CONFIGS}
print("Funciones registradas:", [c[0] for c in CONFIGS])
print("Grids calculados:", list(grids.keys()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, (ax, (nombre, f, _)) in enumerate(zip(axes, CONFIGS)):
    X, Y, Z = grids[nombre]
    cp = ax.contourf(X, Y, np.log1p(Z), levels=60, cmap='viridis')
    plt.colorbar(cp, ax=ax, label='log(1 + f)')
    ax.set_title(f'Figura {i+1}. Contour 2D — {nombre}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.savefig('outputs/contour_2d.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: outputs/contour_2d.png')

In [ ]:
fig = plt.figure(figsize=(13, 5))

for i, (nombre, f, _) in enumerate(CONFIGS):
    X, Y, Z = grids[nombre]           # reutiliza grid ya calculado
    ax = fig.add_subplot(1, 2, i + 1, projection='3d')
    ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9, linewidth=0, antialiased=True)
    ax.set_title(f'Figura {i+3}. Superficie 3D — {nombre}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_zlabel('$f(x_1, x_2)$')

plt.tight_layout()
plt.savefig('outputs/superficie_3d.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: outputs/superficie_3d.png')

## 3. Descenso por gradiente

### 3.1 Algoritmo

Se implementa el descenso por gradiente con **búsqueda de línea por retroceso** (*backtracking line search*), que satisface la condición de Armijo:

$$f(\mathbf{x} + \alpha \mathbf{d}) \leq f(\mathbf{x}) + c \cdot \alpha \cdot \nabla f(\mathbf{x})^\top \mathbf{d}$$

donde $\mathbf{d} = -\nabla f(\mathbf{x})$ es la dirección de descenso, $\alpha$ se reduce por el factor $\rho$ hasta satisfacer la condición.

**Parámetros:** $\alpha_0 = 1.0$, $\rho = 0.5$, $c = 10^{-4}$, tolerancia $\|\nabla f\| < 10^{-6}$, máx. 10 000 iteraciones.

In [ ]:
def backtracking(f, x, direccion, fx, gx_dir):
    """
    Búsqueda de línea por retroceso (condición de Armijo).

    Recibe fx = f(x) y gx_dir = ∇f(x)·d ya calculados en el loop principal
    para evitar evaluaciones redundantes.
    """
    alpha = GD_ALPHA0
    while f(x + alpha * direccion) > fx + GD_C_ARMIJO * alpha * gx_dir:
        alpha *= GD_RHO
        if alpha < 1e-14:
            break
    return alpha


def descenso_gradiente(f, gf, x0):
    """
    Descenso por gradiente con backtracking line search.

    Retorna dict con: x_final, f_final, n_iter, n_evals,
    trayectoria (array N×dim), historial_f (lista).
    """
    x           = np.array(x0, dtype=float)
    trayectoria = [x.copy()]
    historial_f = [f(x)]
    n_evals     = 1

    for i in range(GD_MAX_ITER):
        g = gf(x)
        n_evals += 1
        if np.linalg.norm(g) < GD_TOL:
            break
        direccion = -g
        gx_dir    = np.dot(g, direccion)          # = -||g||² < 0 siempre
        alpha     = backtracking(f, x, direccion, historial_f[-1], gx_dir)
        x         = x + alpha * direccion
        trayectoria.append(x.copy())
        historial_f.append(f(x))
        n_evals += 1

    return {
        'x_final':     x,
        'f_final':     historial_f[-1],   # ya calculado, sin eval extra
        'n_iter':      i + 1,
        'n_evals':     n_evals,
        'trayectoria': np.array(trayectoria),
        'historial_f': historial_f,
    }

### 3.2 Resultados en 2D y 3D

Condición inicial aleatoria uniforme en $[-5, 5]^n$ para cada combinación de función y dimensión.

In [ ]:
resultados_gd = {}
print(f'{"Función":<12} {"Dim":>4} {"f*":>12} {"Iteraciones":>12} {"Evaluaciones":>13}')
print('-' * 56)

SOLO_2D = {'Goldstein-Price', 'Camel 6-hump'}

for nombre, f, gf in CONFIGS:
    resultados_gd[nombre] = {}
    dims = [2] if nombre in SOLO_2D else [2, 3]
    for ndim in dims:
        x0  = np.random.uniform(*BOUNDS_OPT, ndim)
        res = descenso_gradiente(f, gf, x0)
        resultados_gd[nombre][ndim] = res
        print(f'{nombre:<12} {ndim:>4} {res["f_final"]:>12.4e} {res["n_iter"]:>12} {res["n_evals"]:>13}')

### 3.3 Curvas de convergencia

Se grafica $f$ vs. iteración en escala logarítmica para comparar la velocidad de convergencia en 2D y 3D.

In [ ]:
fig, axes  = plt.subplots(1, 2, figsize=(13, 4))
colores_nd = {2: '#1f77b4', 3: '#ff7f0e'}
fig_offset = len(CONFIGS) + 2   # figuras 1-4 ya usadas

for i, (ax, (nombre, f, gf)) in enumerate(zip(axes, CONFIGS)):
    for ndim in [2, 3]:
        hist = resultados_gd[nombre][ndim]['historial_f']
        ax.semilogy(hist, color=colores_nd[ndim], label=f'{ndim}D')
    ax.set_title(f'Figura {fig_offset + i + 1}. Convergencia GD — {nombre}')
    ax.set_xlabel('Iteración')
    ax.set_ylabel('$f$ (escala log)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/convergencia_gd.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: outputs/convergencia_gd.png')

### 3.4 Analisis estadistico: n = 100, 500 y 1000 condiciones iniciales

Para evaluar la robustez del descenso por gradiente se repite el proceso con
**n condiciones iniciales aleatorias** (n = 100, 500 y 1000).

Se registran dos metricas por corrida:
- **f\*** — valor de la funcion objetivo al converger.
- **Evaluaciones** — numero de evaluaciones de la funcion objetivo hasta la convergencia.

Los histogramas permiten ver si el GD encuentra el minimo global de forma
consistente o queda atrapado en minimos locales segun la funcion.

In [ ]:
# ---------------------------------------------------------------------------
# Experimento estadistico: n repeticiones con condicion inicial aleatoria.
# Se usa CONFIGS_ND porque Goldstein y Camel tienen dominios diferentes.
# ---------------------------------------------------------------------------

N_REPS_LIST = [100, 500, 1000]

# resultados_stat[nombre][ndim][n_reps] = {'f_vals': [...], 'evals': [...]}
resultados_stat = {}

for nombre, f, gf in CONFIGS_ND:
    resultados_stat[nombre] = {}
    for ndim in [2, 3]:
        resultados_stat[nombre][ndim] = {}
        for n_reps in N_REPS_LIST:
            np.random.seed(SEED)
            f_vals, evals_list = [], []
            for _ in range(n_reps):
                x0  = np.random.uniform(*BOUNDS_OPT, ndim)
                res = descenso_gradiente(f, gf, x0)
                f_vals.append(res['f_final'])
                evals_list.append(res['n_evals'])
            resultados_stat[nombre][ndim][n_reps] = {
                'f_vals': f_vals,
                'evals':  evals_list,
            }
        print(f"  {nombre} {ndim}D: listo")

print("\nExperimento estadistico completo.")

In [ ]:
# ---------------------------------------------------------------------------
# Histogramas de f* y de evaluaciones de la funcion objetivo.
# Una figura por dimension (2D / 3D).
# Filas = funciones, columnas = n_reps.
# ---------------------------------------------------------------------------

colores_n = {100: '#1f77b4', 500: '#2ca02c', 1000: '#d62728'}

for ndim in [2, 3]:
    n_funcs = len(CONFIGS_ND)
    n_cols  = len(N_REPS_LIST)

    # -- Histogramas de f* --------------------------------------------------
    fig, axes = plt.subplots(n_funcs, n_cols,
                             figsize=(5 * n_cols, 3.5 * n_funcs),
                             constrained_layout=True)
    fig.suptitle(f'Histogramas del valor final f* — GD ({ndim}D)', fontsize=13)

    for row, (nombre, _, __) in enumerate(CONFIGS_ND):
        for col, n_reps in enumerate(N_REPS_LIST):
            ax   = axes[row][col]
            data = resultados_stat[nombre][ndim][n_reps]['f_vals']
            ax.hist(data, bins=30, color=colores_n[n_reps],
                    edgecolor='white', alpha=0.85)
            mediana = float(np.median(data))
            ax.axvline(mediana, color='black', linestyle='--', lw=1.2,
                       label=f'Mediana: {mediana:.2e}')
            ax.set_title(f'{nombre}  n={n_reps}', fontsize=9)
            ax.set_xlabel('f* final', fontsize=8)
            ax.set_ylabel('Frecuencia', fontsize=8)
            ax.legend(fontsize=7)

    fname = f'outputs/histogramas_f_gd_{ndim}d.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Guardado: {fname}')

    # -- Histogramas de evaluaciones ----------------------------------------
    fig, axes = plt.subplots(n_funcs, n_cols,
                             figsize=(5 * n_cols, 3.5 * n_funcs),
                             constrained_layout=True)
    fig.suptitle(f'Histogramas de evaluaciones de f — GD ({ndim}D)', fontsize=13)

    for row, (nombre, _, __) in enumerate(CONFIGS_ND):
        for col, n_reps in enumerate(N_REPS_LIST):
            ax   = axes[row][col]
            data = resultados_stat[nombre][ndim][n_reps]['evals']
            ax.hist(data, bins=30, color=colores_n[n_reps],
                    edgecolor='white', alpha=0.85)
            mediana = float(np.median(data))
            ax.axvline(mediana, color='black', linestyle='--', lw=1.2,
                       label=f'Mediana: {mediana:.0f}')
            ax.set_title(f'{nombre}  n={n_reps}', fontsize=9)
            ax.set_xlabel('Evaluaciones de f', fontsize=8)
            ax.set_ylabel('Frecuencia', fontsize=8)
            ax.legend(fontsize=7)

    fname = f'outputs/histogramas_evals_gd_{ndim}d.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Guardado: {fname}')

## 4. Animaciones GIF

Se anima la trayectoria del descenso por gradiente sobre el contour plot 2D.
Cada frame muestra la posición actual y la ruta recorrida hasta ese momento.

Se genera un **GIF por funcion** (6 en total) mostrando la trayectoria sobre el contour 2D.

In [ ]:
def animar_descenso(f, gf, nombre, seed=SEED):
    """Genera GIF de la trayectoria del GD sobre el contour 2D."""
    np.random.seed(seed)
    res  = descenso_gradiente(f, gf, np.random.uniform(*BOUNDS_OPT, 2))
    tray = res['trayectoria']
    hist = res['historial_f']

    X, Y, Z = grids[nombre]          # reutiliza grid ya calculado
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.contourf(X, Y, np.log1p(Z), levels=60, cmap='viridis')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_xlim(BOUNDS_2D)
    ax.set_ylim(BOUNDS_2D)

    linea,  = ax.plot([], [], 'w-', lw=1.5, alpha=0.85)
    punto,  = ax.plot([], [], 'ro', ms=7)
    titulo  = ax.set_title('')
    indices = np.linspace(0, len(tray) - 1, GIF_FRAMES, dtype=int)

    def actualizar(k):
        idx = indices[k]
        linea.set_data(tray[:idx+1, 0], tray[:idx+1, 1])
        punto.set_data([tray[idx, 0]], [tray[idx, 1]])
        titulo.set_text(f'GD — {nombre} | iter {idx} | f = {hist[min(idx, len(hist)-1)]:.3e}')
        return linea, punto, titulo

    ani = animation.FuncAnimation(
        fig, actualizar, frames=GIF_FRAMES,
        init_func=lambda: (linea, punto, titulo), blit=True
    )
    ruta = 'outputs/gd_' + nombre.lower().replace(' ', '_') + '_2d.gif'
    ani.save(ruta, writer='pillow', fps=GIF_FPS)
    plt.close()
    print(f'Guardado: {ruta}  ({Path(ruta).stat().st_size // 1024} KB)')
    return ruta


for nombre, f, gf in CONFIGS:
    display(Image(animar_descenso(f, gf, nombre)))

## 5. Conclusiones

- **Rosenbrock:** el descenso por gradiente converge al mínimo global $f=0$ en 2D y 3D, pero requiere muchas iteraciones por el valle estrecho y curvado.

- **Rastrigin:** el gradiente queda atrapado en mínimos locales con alta frecuencia; el resultado depende fuertemente de la condición inicial. Esto motiva el uso de métodos heurísticos.

- **Costo computacional:** el número de evaluaciones es bajo para Rosenbrock (convergencia limpia), pero engañoso para Rastrigin, ya que el óptimo encontrado no es necesariamente el global.

> El análisis comparativo completo contra EA, PSO y DE se presenta en el **Notebook 02**.